# Quick Classification Run (No Topics / No Sentiment)

Lightweight notebook that **only loads the interaction + role classifiers** — skips spaCy, sentence-transformers, and VADER entirely for faster startup and execution.

## Configuration

In [ ]:
# --- Edit these as needed ---
START_DATE = "2026-01-15"
END_DATE   = "2026-03-26"
COMPANY_FILE = "tickers.csv"   # or set to None and use COMPANIES list
COMPANIES = None               # e.g. ["AAPL", "MSFT"] — overrides COMPANY_FILE if set

## 1. Setup (minimal imports — no spaCy, no sentence-transformers)

In [ ]:
import os, re, time, warnings
from datetime import datetime

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import torch
torch.set_num_threads(2)
warnings.filterwarnings('ignore', message='.*incorrect regex pattern.*', category=FutureWarning)

import pandas as pd
import google.auth
from google.cloud import bigquery
from transformers import pipeline
from tqdm.notebook import tqdm

# BigQuery config
BQ_PROJECT   = "sri-benchmarking-databases"
BQ_DATASET   = "pressure_monitoring"
BQ_SOURCE    = f"{BQ_PROJECT}.{BQ_DATASET}.earnings_call_transcript_content"
BQ_METADATA  = f"{BQ_PROJECT}.{BQ_DATASET}.earnings_call_transcript_metadata"
BQ_CORP_REF  = f"{BQ_PROJECT}.{BQ_DATASET}.corporation_reference"

INTERACTION_ID_MAP = {"LABEL_0": "Admin", "LABEL_1": "Answer", "LABEL_2": "Question"}
ROLE_ID_MAP = {"LABEL_0": "Admin", "LABEL_1": "Analyst", "LABEL_2": "Executive", "LABEL_3": "Operator"}

print("Setup complete.")

## 2. Load classifiers only (skip embedder, spaCy, VADER)

In [ ]:
%%time
model_dir = os.path.join(os.getcwd(), "models")

print("Loading interaction classifier...")
interaction_classifier = pipeline("text-classification", model=os.path.join(model_dir, "eng_type_class_v1"))

print("Loading role classifier...")
role_classifier = pipeline("text-classification", model=os.path.join(model_dir, "role_class_v1"))

print("Models loaded.")

## 3. Load companies & query BigQuery

In [ ]:
%%time
# Load companies
if COMPANIES:
    company_symbols = [s.strip().upper() for s in COMPANIES]
else:
    ticker_df = pd.read_csv(COMPANY_FILE).dropna(subset=['symbol'])
    company_symbols = ticker_df['symbol'].tolist()
print(f"Companies: {len(company_symbols)}")

# Auth with Drive scope (required for Drive-backed BQ tables)
credentials, _ = google.auth.default(
    scopes=[
        'https://www.googleapis.com/auth/cloud-platform',
        'https://www.googleapis.com/auth/drive',
    ]
)
client = bigquery.Client(project=BQ_PROJECT, credentials=credentials)

# Build and run query
companies_str = "', '".join(company_symbols)
query = f"""
    SELECT
        t.transcript_id, t.paragraph_number, t.speaker, t.content,
        m.* EXCEPT(transcript_id),
        cr.corporation, cr.sector
    FROM `{BQ_SOURCE}` t
    JOIN `{BQ_METADATA}` m ON t.transcript_id = m.transcript_id
    LEFT JOIN `{BQ_CORP_REF}` cr ON m.symbol = cr.Ticker
    WHERE m.symbol IN ('{companies_str}')
      AND m.report_date >= '{START_DATE}'
      AND m.report_date <= '{END_DATE}'
    ORDER BY m.report_date DESC, m.symbol, t.paragraph_number
"""

print("Running BigQuery...")
df = client.query(query).result().to_dataframe()
print(f"Fetched {len(df)} rows, {df['transcript_id'].nunique()} transcripts")

## 4. Preprocess: ensure complete transcripts, rejoin fragments, strip operator intros

In [ ]:
%%time
# Ensure complete transcripts
complete_ids = []
for tid in df['transcript_id'].unique():
    paras = sorted(df.loc[df['transcript_id'] == tid, 'paragraph_number'].unique())
    expected = list(range(paras[0], paras[0] + len(paras)))
    if paras == expected:
        complete_ids.append(tid)
df = df[df['transcript_id'].isin(complete_ids)]
print(f"Complete transcripts: {len(complete_ids)}, segments: {len(df)}")

# Rejoin fragments
rejoined = []
current = df.iloc[0].to_dict()
for i in range(1, len(df)):
    nxt = df.iloc[i].to_dict()
    msg = str(current['content']).strip()
    is_fragment = not any(msg.endswith(p) for p in ['.', '?', '!', '"', '\u201c', '\u201d'])
    if nxt['transcript_id'] == current['transcript_id'] and is_fragment:
        current['content'] = msg + " " + str(nxt['content']).strip()
    else:
        rejoined.append(current)
        current = nxt
rejoined.append(current)
df = pd.DataFrame(rejoined)
print(f"After rejoin: {len(df)} segments")

# Strip operator intros
operator_intro_re = r"^(?:We'll|We will|Let's|Certainly\.?)?\s*(?:go ahead and\s+)?(?:take|move to|now go to)\s+(?:our\s+)?(?:the\s+)?(?:first|next)?\s*question\s+(?:from|is from)\s+[^.!?]+[.!?]\s+"
texts = []
cleaned = 0
for t in df['content'].astype(str):
    c = re.sub(operator_intro_re, '', t, flags=re.IGNORECASE).strip()
    if len(c) < len(t):
        cleaned += 1
    texts.append(c if c else t)
print(f"Cleaned operator intros from {cleaned} segments")

## 5. Classify interaction type + speaker role

In [ ]:
%%time
truncated = [t[:512] for t in texts]

print(f"Classifying {len(truncated)} segments...")
int_results  = interaction_classifier(truncated, batch_size=8)
role_results = role_classifier(truncated, batch_size=8)
print("Classification complete.")

## 6. Assemble results (with post-processing + session grouping)

In [ ]:
%%time
session_re = re.compile(
    r"next question|first question|question (?:is |will be )?(?:coming )?from|"
    r"(?:we'll |we will )?(?:now )?(?:go to|take)|"
    r"move to the line of|go to the line of|from the line of",
    re.IGNORECASE
)

QUESTION_INDICATORS = [
    "have two", "have a question", "wondering", "curious", "want to ask",
    "can you", "could you", "would you", "will you",
    "how do", "how does", "how should", "how would",
    "what", "why", "when", "where", "which",
    "talk about", "comment on", "thoughts on", "perspective on"
]

rows = []
session_id = 0
last_tid = None
prev_int, prev_role = None, None

for i, (_, row) in enumerate(tqdm(list(df.iterrows()), desc="Assembling")):
    # Reset on new transcript
    if last_tid is not None and row['transcript_id'] != last_tid:
        session_id = 0
        prev_int, prev_role = None, None
    last_tid = row['transcript_id']

    text = texts[i]
    interaction_type = INTERACTION_ID_MAP.get(int_results[i]['label'], int_results[i]['label'])
    role_label = ROLE_ID_MAP.get(role_results[i]['label'], role_results[i]['label'])

    # Answer validation
    if interaction_type == "Answer":
        if prev_int != "Question" or prev_role != "Analyst":
            interaction_type = "Admin"

    # Analyst question boost
    if role_label == "Analyst" and interaction_type != "Question":
        lower = text.lower()
        if any(ind in lower for ind in QUESTION_INDICATORS):
            interaction_type = "Question"

    # Session tracking
    lower = text.lower()
    is_operator = role_label == "Operator"
    if (is_operator and any(k in lower for k in ["question", "line of", "analyst"])) or session_re.search(lower):
        session_id += 1

    res = {
        "transcript_id": row['transcript_id'],
        "paragraph_number": row['paragraph_number'],
        "speaker": row['speaker'],
        "qa_session_id": session_id,
        "interaction_type": interaction_type,
        "role": role_label,
        "content": text,
    }
    # Add metadata columns
    for col in row.index:
        if col not in ('transcript_id', 'paragraph_number', 'speaker', 'content'):
            res[col] = row[col]
    rows.append(res)

    prev_int, prev_role = interaction_type, role_label

results_df = pd.DataFrame(rows)
print(f"Assembled {len(results_df):,} rows")

## 7. Save & preview

In [ ]:
os.makedirs("outputs", exist_ok=True)
output_path = f"outputs/quick_classification_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
results_df.to_csv(output_path, index=False)

size_mb = os.path.getsize(output_path) / (1024**2)
print(f"Saved to {output_path} ({size_mb:.2f} MB)")
print(f"  Rows: {len(results_df):,}")
print(f"  Transcripts: {results_df['transcript_id'].nunique()}")
print(f"  Role distribution:")
print(results_df['role'].value_counts().to_string(header=False))
print(f"\n  Interaction type distribution:")
print(results_df['interaction_type'].value_counts().to_string(header=False))

results_df.head(10)